#**Diferença Estrutural: Item-Based vs. User-Based**

Para que o algoritmo KNN calcule a similaridade entre filmes (e não entre usuários), a matriz precisa ser organizada com os filmes nas linhas e os usuários nas colunas:

## **Filtragem por Usuário (User-Based):**

* Linhas = Usuários

* Colunas = Filmes

**O KNN busca usuários similares ao usuário alvo.**

## **Filtragem por Item (Item-Based):**

* **Linhas = Filmes**
* **Colunas = Usuários**

O KNN busca filmes cujos padrões de avaliação ao longo dos 1000 usuários sejam semelhantes.

# **Definindo a base de dados.**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# 1. Configuração de Reprodutibilidade
np.random.seed(42)

# 2. Definição do Escopo
num_usuarios = 1000
filmes = [
    "O Padrinho",
    "Matrix",
    "Inception",
    "Interstellar",
    "Titanic",
    "Pulp Fiction",
    "Forrest Gump",
    "Vingadores: Ultimato",
    "O Senhor dos Anéis",
    "Coringa",
]

# 3. Geração da Matriz de Avaliações
# Criamos uma matriz de dimensões (10 filmes x 1000 usuários)
notas_possiveis = np.arange(0, 5.5, 0.5)
matriz_notas = np.random.choice(
    notas_possiveis, size=(len(filmes), num_usuarios)
)

# Simulação de Sparsity: ~20% das notas são ausentes (NaN)
mascara_faltantes = np.random.rand(len(filmes), num_usuarios) < 0.20
matriz_notas[mascara_faltantes] = np.nan

# 4. Construção do DataFrame com os Filmes no Índice
df_itens = pd.DataFrame(
    matriz_notas,
    index=filmes,
    columns=[f"Usuario_{i+1:04d}" for i in range(num_usuarios)],
)

print("--- Formato da Base de Dados Baseada em Itens (Primeiros 3 Usuários) ---")
print(df_itens.iloc[:, :3])

--- Formato da Base de Dados Baseada em Itens (Primeiros 3 Usuários) ---
                      Usuario_0001  Usuario_0002  Usuario_0003
O Padrinho                     3.0           1.5           5.0
Matrix                         NaN           NaN           NaN
Inception                      1.0           2.0           2.0
Interstellar                   2.0           NaN           1.5
Titanic                        2.5           NaN           2.0
Pulp Fiction                   0.0           4.0           3.0
Forrest Gump                   2.0           4.0           NaN
Vingadores: Ultimato           0.5           3.5           3.5
O Senhor dos Anéis             2.0           NaN           2.5
Coringa                        NaN           0.0           4.0


# **Pré-Processamento**

In [ ]:
# Na abordagem Item-Based, preenchemos os NaNs com a média do próprio filme (média da linha)
df_itens_imputado = df_itens.apply(lambda row: row.fillna(row.mean()), axis=1)

# **Definindo o algoritmo**

In [ ]:
# Usamos métrica de Cosseno para medir o ângulo entre os vetores de cada filme
knn_item = NearestNeighbors(n_neighbors=4, metric="cosine", algorithm="brute")
knn_item.fit(df_itens_imputado)


NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=4)

# **Processamento**

In [ ]:
filme_alvo = "Matrix"
vetor_filme = df_itens_imputado.loc[[filme_alvo]]

distancias, indices = knn_item.kneighbors(vetor_filme)

print(f"\n--- Filmes mais semelhantes ao filme '{filme_alvo}' ---")
for idx, dist in zip(indices[0][1:], distancias[0][1:]):
    nome_filme = df_itens_imputado.index[idx]
    similaridade = 1 - dist
    print(
        f"Filme: {nome_filme:<20} | Similaridade de Cosseno: {similaridade:.4f}"
    )


--- Filmes mais semelhantes ao filme 'Matrix' ---
Filme: Pulp Fiction         | Similaridade de Cosseno: 0.7662
Filme: Titanic              | Similaridade de Cosseno: 0.7589
Filme: Coringa              | Similaridade de Cosseno: 0.7589
